# Analyse accélérométrique de la marche : sujet sain vs sujet avec prothèse

Ce notebook calcule la **RMS (valeur efficace) de l'accélération sur les 3 axes** pendant la marche,
pour plusieurs positions de capteur (cou, épaules, lombaires), et compare un sujet sain à un sujet avec prothèse.

**Utilisation** : placer les 4 fichiers CSV dans `data/input/`, vérifier la section *Configuration*,
puis exécuter toutes les cellules (*Run All*). Les résultats sont enregistrés dans `data/output/`.

Formule : `RMS = sqrt( (1/N) × Σ (aᵢ − ā)² )` par axe ; résultante : `sqrt(RMS_x² + RMS_y² + RMS_z²)`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import signal

## 1. Configuration

Seule cette section est à adapter. Les chemins sont **relatifs** : le notebook fonctionne sur n'importe quel ordinateur,
tant qu'il est lancé depuis la racine du projet avec les CSV dans `data/input/`.

In [ ]:
# --- Dossiers (chemins relatifs) ---
DOSSIER_ENTREE = Path("data/input")
DOSSIER_SORTIE = Path("data/output")

# --- Fichiers d'entrée ---
# groupe A : cou -> épaule droite -> lombaire droite
# groupe B : lombaires (centre) -> épaule gauche -> lombaire gauche
FICHIERS = {
    "sain":     {"A": "sain_A.csv",     "B": "sain_B.csv"},
    "prothese": {"A": "prothese_A.csv", "B": "prothese_B.csv"},
}

# --- Format des CSV (sans en-tête) : date-heure, x, y, z ---
COLONNE_TEMPS = "temps"
COLONNES_AXES = ["x", "y", "z"]

# --- Fenêtres de marche (HH:MM:SS) : 2 allers-retours par position ---
MARCHE_1 = [("10:02:40", "10:04:25"), ("10:07:25", "10:09:02")]
MARCHE_2 = [("10:15:32", "10:17:08"), ("10:18:22", "10:19:56")]
MARCHE_3 = [("10:26:40", "10:28:15"), ("10:32:46", "10:34:13")]

PROTOCOLE = {
    "A": [("cou", MARCHE_1),
          ("epaule droite", MARCHE_2),
          ("lombaire droite", MARCHE_3)],
    "B": [("lombaires centre", MARCHE_1),
          ("epaule gauche", MARCHE_2),
          ("lombaire gauche", MARCHE_3)],
}

# --- Prétraitement ---
FILTRE = False       # False : retrait de la moyenne ; True : passe-bande 0,5-20 Hz
DUREE_MIN_S = 1.0    # une fenêtre contenant moins de données est ignorée

COLS_RES = ["x", "y", "z", "resultante"]

## 2. Fonctions

- `temps_en_secondes` : convertit la colonne date-heure en secondes depuis minuit ;
- `preparer` : retire la moyenne (ou applique un passe-bande de Butterworth à phase nulle) ;
- `ligne` : calcule la RMS par axe et la résultante ;
- `analyser_fichier` : isole les fenêtres de marche et calcule les RMS pour chaque position.

In [ ]:
def en_secondes(hhmmss):
    h, m, s = hhmmss.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)


def temps_en_secondes(serie):
    """Convertit la colonne date-heure en secondes écoulées depuis minuit."""
    dt = pd.to_datetime(serie, format="ISO8601", errors="coerce")
    if dt.isna().all():
        dt = pd.to_datetime(serie, errors="coerce")
    if dt.isna().all():
        raise ValueError("Colonne de temps illisible (format attendu : AAAA-MM-JJ HH:MM:SS.mmm).")
    return (dt.dt.hour * 3600 + dt.dt.minute * 60 + dt.dt.second
            + dt.dt.microsecond / 1e6).to_numpy()


def preparer(data, fs, filtre):
    """Centrage (a - moyenne) ou passe-bande de Butterworth 0,5-20 Hz (ordre 4, phase nulle)."""
    if filtre:
        nyq = fs / 2
        haut = min(20.0, 0.9 * nyq)
        b, a = signal.butter(4, [0.5 / nyq, haut / nyq], btype="band")
        return signal.filtfilt(b, a, data, axis=0)
    return data - data.mean(axis=0)


def ligne(capteur, fenetre, x):
    """RMS par axe = sqrt(mean(x^2)) sur le signal centré, et RMS résultante."""
    r = np.sqrt(np.mean(x ** 2, axis=0))
    return {"capteur": capteur, "fenetre": fenetre,
            "x": r[0], "y": r[1], "z": r[2],
            "resultante": np.sqrt(np.sum(r ** 2)),
            "n_echantillons": len(x)}


def analyser_fichier(chemin, protocole, filtre):
    df = pd.read_csv(chemin, header=None, names=[COLONNE_TEMPS] + COLONNES_AXES)
    t = temps_en_secondes(df[COLONNE_TEMPS])
    fs = 1.0 / np.median(np.diff(t))
    data = df[COLONNES_AXES].to_numpy(dtype=float)
    print(f"{chemin.name} : fs estimée = {fs:.1f} Hz, {len(df)} échantillons")

    lignes = []
    for capteur, fenetres in protocole:
        morceaux = []
        for k, (debut, fin) in enumerate(fenetres, 1):
            masque = (t >= en_secondes(debut)) & (t <= en_secondes(fin))
            if masque.sum() < DUREE_MIN_S * fs:
                print(f"  ATTENTION : fenêtre {debut}-{fin} ({capteur}) vide ou trop courte, ignorée.")
                continue
            x = preparer(data[masque], fs, filtre)
            morceaux.append(x)
            lignes.append(ligne(capteur, f"fenetre {k}", x))
        if morceaux:  # RMS sur l'ensemble des 2 fenêtres de la position
            lignes.append(ligne(capteur, "total", np.vstack(morceaux)))
    return pd.DataFrame(lignes)

## 3. Calcul sur les 4 fichiers

Pour chaque fenêtre de marche, puis pour l'ensemble des deux fenêtres d'une position (ligne `total`).
Le détail est enregistré dans `data/output/rms_resultats.csv`.

In [ ]:
morceaux = []
for sujet, groupes in FICHIERS.items():
    for groupe, nom in groupes.items():
        res = analyser_fichier(DOSSIER_ENTREE / nom, PROTOCOLE[groupe], FILTRE)
        res.insert(0, "sujet", sujet)
        morceaux.append(res)
tout = pd.concat(morceaux, ignore_index=True)

DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
tout.to_csv(DOSSIER_SORTIE / "rms_resultats.csv", index=False)
tout.round(4)

## 4. Comparaison sain / prothèse par position

RMS calculée sur l'ensemble des deux allers-retours de chaque position, et rapport prothèse / sain.

In [ ]:
totaux = tout[tout["fenetre"] == "total"]
ordre = [c for g in PROTOCOLE.values() for c, _ in g]

sain = totaux[totaux["sujet"] == "sain"].set_index("capteur").loc[ordre, COLS_RES]
prot = totaux[totaux["sujet"] == "prothese"].set_index("capteur").loc[ordre, COLS_RES]
ratio = prot / sain

comparaison = pd.concat({"sain": sain, "prothese": prot, "ratio prothese/sain": ratio}, axis=1)
comparaison.to_csv(DOSSIER_SORTIE / "rms_comparaison.csv")

print("RMS - sujet sain")
display(sain.round(4))
print("RMS - sujet avec prothèse")
display(prot.round(4))
print("Ratio prothèse / sain")
display(ratio.round(2))

In [ ]:
x = np.arange(len(COLS_RES))
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=True)
for ax, c in zip(axes.ravel(), ordre):
    ax.bar(x - 0.2, sain.loc[c], 0.4, label="Sain")
    ax.bar(x + 0.2, prot.loc[c], 0.4, label="Prothèse")
    ax.set_xticks(x)
    ax.set_xticklabels(COLS_RES)
    ax.set_title(c.capitalize())
axes[0, 0].set_ylabel("RMS (g)")
axes[1, 0].set_ylabel("RMS (g)")
axes[0, 0].legend()
plt.tight_layout()
plt.savefig(DOSSIER_SORTIE / "rms_comparaison.png", dpi=150)
plt.show()